In [1]:
from src.server.models.scraper_model import ScraperModel
from src.runtime_constants import rr
from src.server.models.gpt_tagger.prompts_configs import PromptsConfigs

from src.server.models.user_quota_model import QuotaPortalInternalBatchContext, QuotaPortalUserContext
from src.db.mysql import DbConnection

#conn = DbConnection(rr.ENV_PROD, is_remote=True)
conn = DbConnection()

quota_context = QuotaPortalInternalBatchContext(QuotaPortalUserContext.QUOTA_OPENAI, "tag app")

while True:

    row = conn.select_one_or_none("""
    SELECT e.app_id_int, m.app_id_in_store, m.store
      FROM app_temp_data.erik_genres_topics_tags e,
           app.map_app_id_to_store_id m
     WHERE e.state = 'queue'
       AND e.app_id_int = m.app_id_int
     ORDER BY e.order_by DESC
     LIMIT 1
    """)

    if row is None:
        break

    app_id_int = row["app_id_int"]
    print(f"App id: {app_id_int}")

    conn.query("""
    UPDATE app_temp_data.erik_genres_topics_tags e
       SET e.t = NOW(),
           e.state = 'working'
     WHERE e.app_id_int = %s
    """, [app_id_int])
    conn.commit()

    try:
        res = ScraperModel.scrap_app(conn=conn, app_id_in_store=row["app_id_in_store"], store=row["store"])

        if res["state"] == 1:

            print(f"App id: {app_id_int} scraped")

            res = PromptsConfigs(conn=conn).tag_prompt_with_ranking(
                quota_context=quota_context,
                app_id_int=app_id_int,
                prompt_id="gpt4_genres",
                prompt_rank_id="gpt4_genres_prio"
            )
            print(f"App id: {app_id_int} {res}")

            if res == "done":
                res = PromptsConfigs(conn=conn).tag_prompt_with_ranking(
                    quota_context=quota_context,
                    app_id_int=app_id_int,
                    prompt_id="gpt4_topics_only",
                    prompt_rank_id="gpt4_topics_prio"
                )
                print(f"App id: {app_id_int} {res}")
                if res == "done":
                    conn.query("""
                    UPDATE app_temp_data.erik_genres_topics_tags e
                       SET e.t = NOW(),
                           e.state = 'done'
                     WHERE e.app_id_int = %s
                    """, [app_id_int])
                    conn.commit()
                else:
                    conn.query("""
                    UPDATE app_temp_data.erik_genres_topics_tags e
                       SET e.t = NOW(),
                           e.state = 'error'
                     WHERE e.app_id_int = %s
                    """, [app_id_int])
                    conn.commit()
            else:
                conn.query("""
                UPDATE app_temp_data.erik_genres_topics_tags e
                   SET e.t = NOW(),
                       e.state = 'error'
                 WHERE e.app_id_int = %s
                """, [app_id_int])
                conn.commit()
        else:
            conn.query("""
            UPDATE app_temp_data.erik_genres_topics_tags e
               SET e.t = NOW(),
                   e.state = 'error'
             WHERE e.app_id_int = %s
            """, [app_id_int])
            conn.commit()
    except Exception as err:
        conn.query("""
        UPDATE app_temp_data.erik_genres_topics_tags e
           SET e.t = NOW(),
               e.state = 'error'
         WHERE e.app_id_int = %s
        """, [app_id_int])
        conn.commit()

conn.commit()
conn.close()

App id: 129
App id: 129 scraped


KeyboardInterrupt: 

In [2]:
conn.close()